# Quickstart

Seven steps from zero configuration to automated prompt optimization.

In [1]:
from spaneval import evaluate, to_entities
from spaneval.toy_data import toy_true, toy_pred
from spaneval.strategies import Strict, ProportionalCoverage

## Step 1 — First look, zero configuration

You have a ground truth and a set of LLM predictions.
Load the built-in toy dataset and call `evaluate()` and `report()` to get a quick look at the results.

In [2]:
results = evaluate(toy_true, toy_pred)
results.report()

Strategies: Strict, AnyOverlap

  Entity Type    Total    Missed    Spurious    Precision (%)    Recall (%)
      Overall       10         2           2          65 ± 15       65 ± 15
         DATE        3         1           0          50 ± 50       33 ± 33
          ORG        2         0           1          67 ±  0      100 ±  0
       PERSON        5         1           1          70 ± 10       70 ± 10


The table has one row per entity type plus an Overall row. Columns: total true entities, how many were missed (no overlapping prediction), how many predictions were spurious (no overlapping true entity), and precision/recall shown as midpoint ± half-range across two strategies.

The default strategies — Strict (ceiling) and AnyOverlap (floor) — give an instant spread:

- **Strict** = exact span + correct type required
- **AnyOverlap** = any non-zero overlap = full credit, type ignored

A wide ± means predictions frequently overlap true entities but fall short of Strict — either boundaries are off, types are wrong, or both. A narrow ± means predictions are either clearly right or clearly wrong.

## Step 2 — Representing your data

`toy_true` and `toy_pred` are lists of documents; each document is a list of `Entity`
objects. `Entity` has three required fields: `entity_type`, `start`, `end` (character
offsets, half-open interval `[start, end)`).

In [3]:
toy_true[0]
# [
#   Entity(entity_type="PERSON", start= 0, end=10),   # "Anna Weber"
#   Entity(entity_type="ORG",    start=34, end=44),   # "Proxima AG"
#   Entity(entity_type="DATE",   start=48, end=61),   # "12 March 2023"
# ]

[Entity(entity_type='PERSON', start=0, end=10, original_text='Anna Weber', replacement_text=None),
 Entity(entity_type='ORG', start=34, end=44, original_text='Proxima AG', replacement_text=None),
 Entity(entity_type='DATE', start=48, end=61, original_text='12 March 2023', replacement_text=None)]

If your pipeline produces plain dicts, convert them before evaluation using
`to_entities()` (one document) or `to_documents()` (list of documents):

In [4]:
raw_doc = [
    {"entity_type": "PERSON", "start":  0, "end": 10},
    {"entity_type": "ORG",    "start": 34, "end": 44},
]
to_entities(raw_doc)

[Entity(entity_type='PERSON', start=0, end=10, original_text=None, replacement_text=None),
 Entity(entity_type='ORG', start=34, end=44, original_text=None, replacement_text=None)]

If your framework uses different field names, pass `field_names` to remap them — only non-standard fields need to be listed:

In [5]:
# spaCy
to_entities(
    [{"label_": "PERSON", "start_char": 0, "end_char": 10}],
    field_names={"label_": "entity_type", "start_char": "start", "end_char": "end"},
)


[Entity(entity_type='PERSON', start=0, end=10, original_text=None, replacement_text=None)]

In [6]:
# HuggingFace
to_entities(
    [{"entity_group": "PERSON", "start": 0, "end": 10}],
    field_names={"entity_group": "entity_type"},
)


[Entity(entity_type='PERSON', start=0, end=10, original_text=None, replacement_text=None)]

For multi-document data use `to_documents()` and, if necessary, the `field_names` argument.

In [7]:
from spaneval import to_documents

raw_true = [
    [{"entity_type": "PERSON", "start":  0, "end": 10}],
    [{"entity_type": "DATE",   "start": 45, "end": 57}],
]
to_documents(raw_true)

[[Entity(entity_type='PERSON', start=0, end=10, original_text=None, replacement_text=None)],
 [Entity(entity_type='DATE', start=45, end=57, original_text=None, replacement_text=None)]]

## Step 3 — Understand what the spread means

DATE has a wide ± on precision and poor recall regardless of strategy. PERSON is much tighter. Drill into which documents are causing problems.

`missed_docs` and `spurious_docs` are strategy-independent and can be directly fetched from the results.

In [8]:
print("Documents with missed entities:     ", results.missed_docs)
print("Documents with spurious predictions:", results.spurious_docs)

Documents with missed entities:      [1, 2]
Documents with spurious predictions: [3]


`incorrect_docs()` depends on the strategy. Under `Strict`, any overlapping prediction with wrong boundaries or wrong type is flagged. Under `AnyOverlap`, any overlap scores 1.0 — so `incorrect_docs` returns `[]` and only `missed_docs` / `spurious_docs` are non-empty.

In [9]:
from spaneval.strategies import AnyOverlap

print("Incorrect under Strict:     ", results.incorrect_docs(strategy=Strict()))
print("Incorrect under AnyOverlap: ", results.incorrect_docs(strategy=AnyOverlap()))

Incorrect under Strict:      [0, 1, 2]
Incorrect under AnyOverlap:  []


In [10]:
toy_true[0]

[Entity(entity_type='PERSON', start=0, end=10, original_text='Anna Weber', replacement_text=None),
 Entity(entity_type='ORG', start=34, end=44, original_text='Proxima AG', replacement_text=None),
 Entity(entity_type='DATE', start=48, end=61, original_text='12 March 2023', replacement_text=None)]

In [11]:
toy_pred[0]

[Entity(entity_type='PERSON', start=0, end=10, original_text='Anna Weber', replacement_text=None),
 Entity(entity_type='ORG', start=34, end=44, original_text='Proxima AG', replacement_text=None),
 Entity(entity_type='DATE', start=51, end=61, original_text='March 2023', replacement_text=None)]

Inspect those documents to understand whether the problem is span boundaries, wrong types, or missed detections.

## Step 4 — Pin a strategy

Once you understand the spread, pick the strategy that matches what "correct" means for your use case. For anonymization, a slightly off span boundary is usually acceptable — the text is still anonymized. `ProportionalCoverage` (fraction of true-entity characters covered, type ignored) is typically a good choice.

In [12]:
results.report(strategy=ProportionalCoverage())

  Entity Type    Total    Missed    Spurious    Precision (%)    Recall (%)    F1 (%)
      Overall       10         2           2               71            71        71
         DATE        3         1           0               79            53        63
          ORG        2         0           1               67           100        80
       PERSON        5         1           1               71            71        71


You now have exact precision, recall and f1 for one strategy.

## Step 5 — Assign strategies per entity type

Different entity types may warrant different strictness. For anonymization task, names must be found precisely (wrong span could leave part of a name visible), but dates might only need to be found at all.

In [13]:
results.report(
    strategy={"PERSON": Strict(), "DATE": ProportionalCoverage(), "ORG": ProportionalCoverage()},
)

  Entity Type              Strategy    Total    Missed    Spurious    Precision (%)    Recall (%)    F1 (%)
      Overall                 mixed       10         2           2               66            66        66
         DATE  ProportionalCoverage        3         1           0               79            53        63
          ORG  ProportionalCoverage        2         0           1               67           100        80
       PERSON                Strict        5         1           1               60            60        60


The Overall row is a composite: each type contributes its metrics under its own strategy. If any entity type in your data is not covered by the dict and you have not set `default_strategy`, you get an error telling you which types are missing.

Continue in `examples/goals.ipynb` to define targets and automate optimization.

## Step 6 — Define goals

The next question is: what is good enough?
Express this as per-type precision and recall targets using `Goal`.
Both targets are required — 99% recall at 0% precision is not useful anonymization.

In [14]:
from spaneval import Goal

goals = {
    "PERSON": Goal(strategy=Strict(),              recall=0.90, precision=0.80),
    "DATE":   Goal(strategy=ProportionalCoverage(), recall=0.80, precision=0.70),
    "ORG":    Goal(strategy=ProportionalCoverage(), recall=0.85, precision=0.70),
}

results.report_goals(goals)

  Entity Type              Strategy    Recall    R-Target    R-Score    Precision    P-Target    P-Score
      Overall               (goals)                                                              0.66  ←
       PERSON                Strict      0.60        0.90       0.67         0.60        0.80       0.75
         DATE  ProportionalCoverage      0.53        0.80     0.66 ←         0.79        0.70       1.13
          ORG  ProportionalCoverage      1.00        0.85       1.18         0.67        0.70       0.95


Scores are uncapped: > 1.0 means above target. The ← marks the bottleneck -
the weakest type/metric combination. That is what you need to fix first.

## Step 7 — Use `score()` for automated prompt engineering

`score()` returns the bottleneck float without printing — clean for use inside a loop.

In [15]:
score = results.score(goals)
print(f"score = {score:.2f}  (1.0 = all targets exactly met; > 1.0 = all exceeded)")

score = 0.66  (1.0 = all targets exactly met; > 1.0 = all exceeded)


For a full prompt-optimisation workflow using DSPy, see `examples/dspy.ipynb`.